# TradeCoach

> ICT 개념 기반 트레이딩 저널 분석 + 차트 피드백 + 약점 개념 자동 조회 LangGraph 에이전트

## 에이전트 설계 요약

| 레이어 | 노드 | 역할 |
|--------|------|------|
| 입력 | `memory_load_node` | DB에서 과거 약점 / 통계 / 개선 이력 로드 |
| 입력 | `input_router_node` | CSV·이미지·혼합 입력 판별 → 라우팅 |
| 분석 | `journal_analysis_node` | CSV 파싱 → 승률·손익비·드로다운 통계 산출 (GPT-4o-mini) |
| 분석 | `weakness_detect_node` | 임계값 규칙으로 약점 태그 추출 + ICT 개념 조회 |
| 분석 | `chart_analysis_node` | base64 차트 이미지 → ICT 관점 분석 (GPT-4o Vision) |
| 피드백 | `feedback_node` | 차트 피드백 구조화 + 약점 병합 |

**약점 태그 임계값**

| 지표 | 임계값 | 태그 |
|------|--------|------|
| win_rate | < 0.4 | `승률_낮음` |
| avg_rr | < 1.5 | `손익비_부족` |
| max_drawdown | ≥ 3 | `연속손실_패턴` |
| worst_setup | 존재 시 | `{setup명}_개선필요` |

## 그래프 구조

```
START
  └─ memory_load_node
       └─ input_router_node
            ├─ 'journal' ──→ journal_analysis_node
            │                      └─ weakness_detect_node ──→ END
            ├─ 'chart'  ──→ chart_analysis_node
            │                      └─ feedback_node ──→ END
            └─ 'both'   ──→ journal_analysis_node
                                   └─ weakness_detect_node
                                            └─ chart_analysis_node
                                                     └─ feedback_node ──→ END
```

## 1. 환경 설정

In [ ]:
import os
from typing_extensions import TypedDict

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

from db import get_db, init_db

load_dotenv()

MODEL = "openai:gpt-4o"

init_db()
print("DB initialized")

## 2. TradeCoachState

In [ ]:
class TradeCoachState(TypedDict, total=False):
    session_id:      str
    input_type:      str          # 'journal' | 'chart' | 'both'
    journal_data:    str          # CSV 원문
    chart_image:     str          # base64 인코딩 이미지
    stats:           dict         # 승률/손익비/드로다운 등
    weaknesses:      list         # 현재 세션 약점 태그
    past_weaknesses: list         # DB 로드 과거 약점
    chart_feedback:  str          # 차트 분석 결과 텍스트
    current_concept: str          # 현재 학습 중인 개념
    quiz_question:   str          # 퀴즈 문항
    quiz_answer:     str          # 사용자 답변
    quiz_result:     str          # 'pass' | 'fail' | ''
    retry_count:     int          # 퀴즈 재시도 횟수
    trade_count:     int          # 총 분석 트레이드 수
    improvement_log: list         # 세션별 실력 변화
    messages:        list         # 대화 히스토리


DEFAULT_STATE: TradeCoachState = {
    "session_id":      "default",
    "input_type":      "",
    "journal_data":    "",
    "chart_image":     "",
    "stats":           {},
    "weaknesses":      [],
    "past_weaknesses": [],
    "chart_feedback":  "",
    "current_concept": "",
    "quiz_question":   "",
    "quiz_answer":     "",
    "quiz_result":     "",
    "retry_count":     0,
    "trade_count":     0,
    "improvement_log": [],
    "messages":        [],
}

## 3. 노드 함수 정의

### 3-1. memory_load_node · input_router_node

| 노드 | 입력 | 출력 |
|------|------|------|
| `memory_load_node` | `session_id` | `past_weaknesses`, `trade_count`, `improvement_log` |
| `input_router_node` | `journal_data`, `chart_image` | `input_type` |

In [ ]:
def memory_load_node(state: TradeCoachState) -> dict:
    session_id = state.get("session_id", "default")
    past_weaknesses = []
    trade_count = 0
    improvement_log = []

    try:
        with get_db() as conn:
            rows = conn.execute(
                "SELECT weakness FROM weaknesses WHERE session_id = ? ORDER BY count DESC",
                (session_id,),
            ).fetchall()
            past_weaknesses = [r["weakness"] for r in rows]

            count_row = conn.execute(
                "SELECT COALESCE(SUM(trade_count), 0) FROM trade_history WHERE session_id = ?",
                (session_id,),
            ).fetchone()
            trade_count = int(count_row[0]) if count_row else 0

            history_rows = conn.execute(
                "SELECT date, win_rate, avg_rr FROM trade_history"
                " WHERE session_id = ? ORDER BY date DESC LIMIT 10",
                (session_id,),
            ).fetchall()
            improvement_log = [dict(r) for r in history_rows]
    except Exception:
        pass

    return {
        "past_weaknesses": past_weaknesses,
        "trade_count":     trade_count,
        "improvement_log": improvement_log,
    }

In [ ]:
def input_router_node(state: TradeCoachState) -> dict:
    has_journal = bool(state.get("journal_data", "").strip())
    has_chart   = bool(state.get("chart_image",  "").strip())

    if has_journal and has_chart:
        input_type = "both"
    elif has_chart:
        input_type = "chart"
    else:
        input_type = "journal"

    return {"input_type": input_type}


def route_after_input(state: TradeCoachState) -> str:
    return state["input_type"]

### 3-2. journal_analysis_node · weakness_detect_node

| 노드 | LLM | 역할 |
|------|-----|------|
| `journal_analysis_node` | GPT-4o-mini | CSV → stats (win_rate / avg_rr / max_drawdown / best·worst_setup) |
| `weakness_detect_node` | 없음 (규칙 기반) | stats + past_weaknesses → weakness 태그 + ICT 개념 조회 |

In [ ]:
from nodes.analysis_nodes import journal_analysis_node, weakness_detect_node

### 3-3. chart_analysis_node · feedback_node

| 노드 | LLM | 역할 |
|------|-----|------|
| `chart_analysis_node` | GPT-4o Vision | base64 이미지 → ICT 관점 분석 텍스트 |
| `feedback_node` | GPT-4o-mini | 분석 결과 구조화 + chart_weaknesses를 weaknesses에 병합 |

In [ ]:
from nodes.chart_nodes import chart_analysis_node, feedback_node

## 4. Tool 정의

`search_ict_concept` — `tools/ict_concepts.json` 로컬 사전에서 weakness 태그에 해당하는 ICT 개념 정의 + 개선 방법을 반환합니다.  
대소문자 무시 및 부분 매칭을 지원합니다 (예: `OrderBlock_개선필요` → `OrderBlock`).

In [ ]:
from tools import search_ict_concept

## 5. 그래프 조립

In [ ]:
def route_after_weakness(state: TradeCoachState) -> str:
    """both 경로: weakness_detect 완료 후 chart_analysis로 진행."""
    return "chart_analysis" if state.get("input_type") == "both" else END


builder = StateGraph(TradeCoachState)

builder.add_node("memory_load",      memory_load_node)
builder.add_node("input_router",     input_router_node)
builder.add_node("journal_analysis", journal_analysis_node)
builder.add_node("weakness_detect",  weakness_detect_node)
builder.add_node("chart_analysis",   chart_analysis_node)
builder.add_node("feedback",         feedback_node)

builder.add_edge(START, "memory_load")
builder.add_edge("memory_load", "input_router")
builder.add_conditional_edges(
    "input_router",
    route_after_input,
    {
        "journal": "journal_analysis",
        "both":    "journal_analysis",
        "chart":   "chart_analysis",
    },
)
builder.add_edge("journal_analysis", "weakness_detect")
builder.add_conditional_edges(
    "weakness_detect",
    route_after_weakness,
    {
        "chart_analysis": "chart_analysis",
        END:              END,
    },
)
builder.add_edge("chart_analysis", "feedback")
builder.add_edge("feedback", END)

graph = builder.compile()
print("Graph compiled")

## 6. 그래프 시각화

In [ ]:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

## 7. 테스트

> `OPENAI_API_KEY` 환경변수가 필요합니다.  
> `chart_image` 없이 journal 경로만 실행합니다 (chart 경로는 실제 base64 이미지 필요).

In [ ]:
test_journal = """date,setup,result,rr
2026-05-01,FVG,win,2.1
2026-05-02,FVG,loss,-1.0
2026-05-03,OrderBlock,win,1.8
2026-05-04,OrderBlock,loss,-1.0
2026-05-05,FVG,loss,-1.0
2026-05-06,FVG,loss,-1.0"""

In [ ]:
result = graph.invoke({
    **DEFAULT_STATE,
    "session_id":   "test_user",
    "journal_data": test_journal,
})

print("=" * 60)
print("[통계]")
for k, v in result["stats"].items():
    print(f"  {k:15s}: {v}")

print()
print("[약점 태그]")
for w in result["weaknesses"]:
    print(f"  • {w}")

print()
print("[1순위 약점 개념 설명]")
if result["messages"]:
    print(result["messages"][-1].content)
else:
    print("  (약점 없음)")
print("=" * 60)